In [ ]:
%run "./00_CC_Mapping_Setup.ipynb" 

In [ ]:
%sql
/* ===================================================================================
   METRIC: TP01 - TP Jurisdictions
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. TP Unit Mapping: hive_metastore.ra_adido_2025.third_party_unit_mapping
   3. Country Risk Ratings: hive_metastore.ra_adido_2025.td_country_risk_rating_abac
   4. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   
   BUSINESS QUESTION: 
   What number of third party relationships with third parties are in jurisdictions 
   considered a high risk for bribery and corruption during the assessment period?
   
   LOGIC SUMMARY:
   1. MASTER AU ANCHOR: Uses the new Master AU List + TP Unit Mapping, strictly 
      filtered by 'Jurisdiction' for Canada, Hong Kong, and Barbados.
   2. HIGH RISK EVALUATION: Checks three columns against the ABAC High Risk list. 
   3. TPRM STRING MAPPING: Maps TP engagements to AU IDs using string matching.
   4. DEDUPLICATION: Uses ROW_NUMBER() to guarantee each Engagement maps to exactly 
      ONE Assessable Unit, prioritizing matches in 'owning_lob' over 'lob_using'.
   5. FINAL OUTPUT: Strict 6-column structure, converting NULL sums to '0'.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment 
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
),

Mapped_AUs AS (
    SELECT DISTINCT TRIM(CAST(Assessable_Unit_ID AS STRING)) AS AU_ID 
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
),

All_Base_AUs AS (
    SELECT 
        COALESCE(mast.BusinessID, map.AU_ID) AS Base_AU_ID,
        mast.AU_Name,
        mast.Business_Segment,
        CASE WHEN mast.BusinessID IS NOT NULL THEN 'Yes' ELSE 'No' END AS In_ABAC_AU_List
    FROM Master_AUs mast
    FULL OUTER JOIN Mapped_AUs map 
        ON mast.BusinessID = map.AU_ID
),

High_Risk_Countries AS (
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
),

Third_Party_Risk_Data AS (
    SELECT 
        EngagementNumber,
        owning_lob,
        lob_using,
        location_of_tp,
        country_prod_serv_originates,
        Td_Countries_Impacted,
        CASE WHEN COALESCE(TRIM(location_of_tp), '') = ''
              AND COALESCE(TRIM(country_prod_serv_originates), '') = ''
              AND COALESCE(TRIM(Td_Countries_Impacted), '') = '' THEN 1
             ELSE 0 END AS Is_Missing_Jurisdiction
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
),

High_Risk_Engagements AS (
    SELECT DISTINCT 
        tp.EngagementNumber,
        tp.owning_lob,
        tp.lob_using
    FROM Third_Party_Risk_Data tp
    LEFT JOIN High_Risk_Countries h1 ON UPPER(TRIM(tp.location_of_tp)) = h1.CountryName
    LEFT JOIN High_Risk_Countries h2 ON UPPER(TRIM(tp.country_prod_serv_originates)) = h2.CountryName
    LEFT JOIN High_Risk_Countries h3 ON UPPER(TRIM(tp.Td_Countries_Impacted)) = h3.CountryName
    WHERE h1.CountryName IS NOT NULL
       OR h2.CountryName IS NOT NULL
       OR h3.CountryName IS NOT NULL
       OR tp.Is_Missing_Jurisdiction = 1
),

All_Potential_Matches AS (
    -- Find every possible AU match for an engagement
    SELECT 
        hr.EngagementNumber,
        TRIM(CAST(map.Assessable_Unit_ID AS STRING)) AS Mapped_AU_ID,
        CASE 
            WHEN UPPER(hr.owning_lob) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%' THEN 1
            ELSE 2 
        END AS Match_Priority
    FROM High_Risk_Engagements hr
    INNER JOIN hive_metastore.ra_adido_2025.third_party_unit_mapping map
        ON UPPER(hr.owning_lob) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
        OR UPPER(hr.lob_using) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
),

Deduplicated_Matches AS (
    -- Force 1-to-1 mapping: Pick the best AU match per engagement
    SELECT 
        EngagementNumber,
        Mapped_AU_ID
    FROM (
        SELECT 
            EngagementNumber,
            Mapped_AU_ID,
            ROW_NUMBER() OVER(
                PARTITION BY EngagementNumber 
                ORDER BY Match_Priority ASC, Mapped_AU_ID ASC
            ) as rn
        FROM All_Potential_Matches
    )
    WHERE rn = 1
),

Engagements_By_AU AS (
    -- Count the uniquely assigned engagements per AU
    SELECT 
        Mapped_AU_ID,
        COUNT(EngagementNumber) AS High_Risk_Count
    FROM Deduplicated_Matches
    GROUP BY Mapped_AU_ID
)

-- Final Template
SELECT 
    a.Base_AU_ID AS BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'TP01' AS QuestionID,               
    COALESCE(CAST(e.High_Risk_Count AS STRING), '0') AS Response,
    a.In_ABAC_AU_List
    
FROM All_Base_AUs a
LEFT JOIN Engagements_By_AU e 
    ON a.Base_AU_ID = e.Mapped_AU_ID
ORDER BY a.Base_AU_ID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: TP01 - TP Jurisdictions Drill-Down
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
),

Mapped_AUs AS (
    SELECT DISTINCT TRIM(CAST(Assessable_Unit_ID AS STRING)) AS AU_ID 
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
),

All_Base_AUs AS (
    SELECT 
        COALESCE(mast.BusinessID, map.AU_ID) AS Base_AU_ID,
        mast.AU_Name,
        CASE WHEN mast.BusinessID IS NOT NULL THEN 'Yes' ELSE 'No' END AS In_ABAC_AU_List
    FROM Master_AUs mast
    FULL OUTER JOIN Mapped_AUs map 
        ON mast.BusinessID = map.AU_ID
),

High_Risk_Countries AS (
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
),

Third_Party_Risk_Data AS (
    SELECT 
        EngagementNumber,
        ThirdPartyName,
        owning_lob,
        lob_using,
        location_of_tp,
        country_prod_serv_originates,
        Td_Countries_Impacted,
        CASE WHEN COALESCE(TRIM(location_of_tp), '') = ''
              AND COALESCE(TRIM(country_prod_serv_originates), '') = ''
              AND COALESCE(TRIM(Td_Countries_Impacted), '') = '' THEN 1
             ELSE 0 END AS Is_Missing_Jurisdiction
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
),

All_Potential_Matches AS (
    SELECT 
        tp.EngagementNumber,
        tp.ThirdPartyName,
        tp.owning_lob AS Raw_Col_K_owning_lob,
        tp.lob_using AS Raw_Col_L_lob_using,
        tp.location_of_tp AS Country_1,
        tp.country_prod_serv_originates AS Country_2,
        tp.Td_Countries_Impacted AS Country_3,
        tp.Is_Missing_Jurisdiction,
        map.TPRM_Units AS Successfully_Mapped_String,
        TRIM(CAST(map.Assessable_Unit_ID AS STRING)) AS Mapped_AU_ID,
        CASE 
            WHEN UPPER(tp.owning_lob) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%' THEN 1
            ELSE 2 
        END AS Match_Priority
    FROM Third_Party_Risk_Data tp
    INNER JOIN hive_metastore.ra_adido_2025.third_party_unit_mapping map
        ON UPPER(tp.owning_lob) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
        OR UPPER(tp.lob_using) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
),

Deduplicated_Matches AS (
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER(
            PARTITION BY EngagementNumber 
            ORDER BY Match_Priority ASC, Mapped_AU_ID ASC
        ) as rn
        FROM All_Potential_Matches
    ) WHERE rn = 1
)

SELECT DISTINCT
    a.Base_AU_ID AS Assigned_BusinessID,
    a.AU_Name AS Assigned_AU_Name,
    a.In_ABAC_AU_List,
    d.EngagementNumber,
    d.ThirdPartyName,
    d.Successfully_Mapped_String,
    CASE WHEN d.Match_Priority = 1 THEN 'owning_lob' ELSE 'lob_using' END AS Matched_Column,
    d.Raw_Col_K_owning_lob,
    d.Raw_Col_L_lob_using,
    d.Country_1,
    d.Country_2,
    d.Country_3,
    CASE 
        WHEN d.Is_Missing_Jurisdiction = 1 THEN 'Missing Jurisdiction (Auto High Risk)'
        ELSE 'Matched High Risk ABAC List' 
    END AS Risk_Flag_Reason
    
FROM Deduplicated_Matches d
INNER JOIN All_Base_AUs a ON d.Mapped_AU_ID = a.Base_AU_ID

LEFT JOIN High_Risk_Countries h1 ON UPPER(TRIM(d.Country_1)) = h1.CountryName
LEFT JOIN High_Risk_Countries h2 ON UPPER(TRIM(d.Country_2)) = h2.CountryName
LEFT JOIN High_Risk_Countries h3 ON UPPER(TRIM(d.Country_3)) = h3.CountryName

-- Strictly filter for only the high-risk engagements
WHERE h1.CountryName IS NOT NULL
   OR h2.CountryName IS NOT NULL
   OR h3.CountryName IS NOT NULL
   OR d.Is_Missing_Jurisdiction = 1
   
ORDER BY a.Base_AU_ID;